In [1]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
iris = fetch_ucirepo(id=53) 
  
# data (as pandas dataframes) 
X = iris.data.features 
y = iris.data.targets 
  
# metadata 
print(iris.metadata) 
  
# variable information 
print(iris.variables) 


{'uci_id': 53, 'name': 'Iris', 'repository_url': 'https://archive.ics.uci.edu/dataset/53/iris', 'data_url': 'https://archive.ics.uci.edu/static/public/53/data.csv', 'abstract': 'A small classic dataset from Fisher, 1936. One of the earliest known datasets used for evaluating classification methods.\n', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Tabular'], 'num_instances': 150, 'num_features': 4, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1936, 'last_updated': 'Tue Sep 12 2023', 'dataset_doi': '10.24432/C56C76', 'creators': ['R. A. Fisher'], 'intro_paper': {'ID': 191, 'type': 'NATIVE', 'title': 'The Iris data set: In search of the source of virginica', 'authors': 'A. Unwin, K. Kleinman', 'venue': 'Significance, 2021', 'year': 2021, 'journal': 'Significance, 2021', 'DOI': '1740-9713.01589', 'URL': 'https://www.semanticscholar.org

In [ ]:
'''
1° Passo - Matriz W
  Para cada linha, sortear 3 valores entre 0 e 1, cuja soma deles dá 1 (100%)
  Normalizar (??????????????????????????????????????????)
--> Teremos um peso para cada cluster (coluna --> 3 C1, C2, C3) para cada amostra (linha --> 150)

2° Passo - Calcular o centróide dos cluster
  Pra cada coluna (cluster)
    soma os valores da linha
    Aplica fórmula
    Peso_centroide_x = soma (vetor_linha * peso_linha --> vet*0.7) / som dos pesos (todas linhas)
          cj         =    ∑   xi        *   wij                   /    ∑   wij

    xi = [sepal length, sepal width, petal length, petal width]
    wij = somatória dos pesos
    cj = vetor com 4 entradas --> centroide pra cada variável     [c1, c2, c3, c4]

3° Passo - Atualizar pesos
  Pra cada linha
    Calcula o peso pra cada cluster
    wi1 = xxxxx <-- dist_euclidiana(Xi - c1) / dist_euclidiana(Xi - c1) + dist_euclidiana(Xi - c2) + dist_euclidiana(Xi - c3)
    wi2 = xxxxx
    wi3 = xxxxx


'''

In [89]:
# 1° Passo - Montar Matriz W com sorteio

import random
import pandas as pd
import numpy as np

# Amostra C1 C2 C3
W = []

# Teremos que fazer o sorteio de 3 valores 150 vezes (tamanho da iris)
for linha in range(len(X)):
    # Gerando 3 números aleatórios
    prob = [random.random() for _ in range(3)]

    # Transformando em entre 0 e 1 (percentual)
    prob_normalizados = [x / sum(prob) for x in prob]
    
    W.append(prob_normalizados)
    
W = pd.DataFrame(W, columns=['C1', 'C2', 'C3'])
W

,C1,C2,C3
0,0.023452,0.285903,0.690645
1,0.223403,0.311373,0.465223
2,0.866258,0.008469,0.125273
3,0.276765,0.068031,0.655204
4,0.368743,0.317686,0.313571
...,...,...,...
145,0.618982,0.323605,0.057413
146,0.323456,0.156043,0.520500
147,0.310581,0.308385,0.381033
148,0.301996,0.536262,0.161742


In [84]:
# 2° Passo - Calcular o centróide dos cluster

# Criando o campo xi com todos os dados direto no df
X['xi'] = list(X[['petal length', 'petal width', 'sepal length', 'sepal width']].values)


def calcular_centroides(X, W):
    centroides = []

    # Para cada coluna, calcularemos o centroide
    for col in W.columns:
        sum_xi_wij = sum(xi * peso for xi, peso in zip(X['xi'], W[col]))
        
        sum_wij = W[col].sum()
        
        centroides.append(sum_xi_wij / sum_wij)

    return centroides

In [81]:
# 3° Passo - Atualizar pesos

# Função de distância Euclidiana
def dist(a, b):
    return np.linalg.norm(a - b)

# Função para atualizar os pesos
def att_pesos(centroides, X):
    m = 2 
    novos_pesos = []
    # Pra cada linha, atualizaremos os pesos
    for _, linha in X.iterrows():
        # Extraindo a linha
        xi = linha['xi']
        
        # Calculando a distância de cada centroide para essa linha
        distancias = [dist(xi, c) for c in centroides]
        
        probabilidade_att = []
        for d in distancias:
            # Se a distância for 0, o peso é 1
            if d == 0:
                probabilidade_att.append(1.0)
            else:
                soma_razões = sum([(d / d_2)**(2/(m-1)) for d_2 in distancias])
                probabilidade_att.append(1 / soma_razões)
                
        novos_pesos.append(probabilidade_att)

    # Atualizando a matriz W com os novos pesos calculados
    W = pd.DataFrame(novos_pesos, columns=['C1', 'C2', 'C3'])
    return W

In [ ]:
# Colocar isso em um loop
# Critério de parada?

for i in range(20):
   centroides = calcular_centroides(X, W)
   W = att_pesos(centroides, X)
   print(centroides, '\n')

In [ ]:
# Pendências:
# Como escolhe o m??
# Normalizar? Isso é no sorteio?
# Pensar e implementar um critério de parada